# 선형 시스템 실습: 1차원 기초와 2차원 확장


## 학습 개요


선형 시스템의 상태방정식(State-space representation)은 다음과 같이 표현된다.

본 장은 다음의 2단계 실습으로 구성된다.

1. **Part A (함께 따라하기)**: 1차원 선형 시스템의 기본 구조를 확인한다.
2. **Part B (스스로 해보기)**: 동일한 구조를 2차원 CA 모델로 확장한다.

공통으로 사용하는 선형 시스템 표현은 다음과 같다.

**State Equation (연속/이산)**
- 연속 시간 (Continuous-time):
$$
\dot{\mathbf{X}}(t) = \mathbf{A}\mathbf{X}(t) + \mathbf{B}\mathbf{u}(t)
$$
- 이산 시간 (Discrete-time):
$$
\mathbf{X}_{k+1} = \mathbf{F}\mathbf{X}_k + \mathbf{G}\mathbf{u}_k
$$

**Measurement Equation**
$$
\mathbf{Y}_k = \mathbf{H}\mathbf{X}_k
$$

- $\mathbf{X}_k$: state vector
- $\mathbf{A}$: continuous-time system matrix
- $\mathbf{F}$: discrete-time system matrix
- $\mathbf{u}_k$: input
- $\mathbf{B}$: continuous-time control matrix
- $\mathbf{G}$: discrete-time control matrix
- $\mathbf{Y}_k$: measurement
- $\mathbf{H}$: measurement matrix

### Plot settings (Korean labels)

아래 셀을 **그래프를 그리기 전에 한 번 실행**한다. OS별로 흔한 한글 폰트를 `sans-serif` 우선순위에 넣고, 없으면 `DejaVu Sans`로 넘어간다.

In [ ]:
# Matplotlib: avoid broken Korean in titles/labels
import matplotlib as mpl
import platform

# Prefer OS-default Korean-capable fonts; fall back to DejaVu Sans.
if platform.system() == "Windows":
    mpl.rcParams["font.family"] = "sans-serif"
    mpl.rcParams["font.sans-serif"] = ["Malgun Gothic", "맑은 고딕", "DejaVu Sans"]
elif platform.system() == "Darwin":
    mpl.rcParams["font.family"] = "sans-serif"
    mpl.rcParams["font.sans-serif"] = ["Apple SD Gothic Neo", "AppleGothic", "DejaVu Sans"]
else:
    mpl.rcParams["font.family"] = "sans-serif"
    mpl.rcParams["font.sans-serif"] = ["NanumGothic", "Noto Sans CJK KR", "DejaVu Sans"]

mpl.rcParams["axes.unicode_minus"] = False
print("matplotlib sans-serif:", mpl.rcParams["font.sans-serif"])

## Part A. 함께 따라하기: 1차원 Mass-Spring-Damper 시스템

본 파트는 선형 시스템의 핵심 구조를 익히기 위한 **가이드 실습**이다.

- 시스템: 질량-스프링-댐퍼(Mass-Spring-Damper)
- 상태: 변위 `x`, 속도 `v`
- 입력: 외력 `u` (N)
- 출력: 변위 `y = x`

1차원 동역학 모델에서 상태 전파와 출력 계산 과정을 확인한 후, 동일한 상태공간 표현을 2차원 모델에 적용한다.

### 1차원 모델 식

질량 `m`, 감쇠계수 `c`, 스프링상수 `k`인 시스템의 2계 미분방정식은 다음과 같다.

$$
m\ddot{x}(t) + c\dot{x}(t) + kx(t) = u(t)
$$

먼저 가속도 항을 정리하면

$$
\ddot{x}(t)= -\frac{k}{m}x(t)-\frac{c}{m}\dot{x}(t)+\frac{1}{m}u(t)
$$

상태를 다음과 같이 정의한다.

$$
x_1(t)=x(t), \quad x_2(t)=\dot{x}(t)
$$

그러면 연속시간 1차 연립방정식(상태공간 표현)은 다음과 같다.

$$
\dot{x}_1(t)=x_2(t)
$$
$$
\dot{x}_2(t)= -\frac{k}{m}x_1(t)-\frac{c}{m}x_2(t)+\frac{1}{m}u(t)
$$

이를 행렬로 표현하면 다음과 같다.

$$
\begin{bmatrix}
\dot{x}_1(t) \\
\dot{x}_2(t)
\end{bmatrix} =
\underbrace{
\begin{bmatrix}
0 & 1 \\
-\frac{k}{m} & -\frac{c}{m}
\end{bmatrix}
}_{A}
\begin{bmatrix}
x_1(t) \\
x_2(t)
\end{bmatrix}
+
\underbrace{
\begin{bmatrix}
0 \\
\frac{1}{m}
\end{bmatrix}
}_{B}
u(t)
$$

여기서 A는 시스템의 동특성을 나타내는 행렬(시스템 행렬), B는 입력이 시스템 상태에 어떻게 작용하는지 나타내는 행렬(입력 행렬)이다.

이를 샘플링 시각 `t_k=k\Delta t`에서 Forward Euler로 이산화하면

$$
x_1[k+1]=x_1[k]+\Delta t\,\dot{x}_1[k]=x_1[k]+\Delta t\,x_2[k]
$$
$$
x_2[k+1]=x_2[k]+\Delta t\,\dot{x}_2[k]
= x_2[k] + \Delta t\left(-\frac{k}{m}x_1[k]-\frac{c}{m}x_2[k]+\frac{1}{m}u[k]\right)
= - \frac{k}{m}\Delta t\,x_1[k] + \left(1 - \frac{c}{m}\Delta t\right)x_2[k]  + \frac{\Delta t}{m}u[k]
$$

상태벡터를

$$
\mathbf{x}_k = \begin{bmatrix}x_k \\ v_k\end{bmatrix}
$$

로 두면 이산시간 상태방정식은

$$
\mathbf{x}_{k+1} =
\underbrace{\begin{bmatrix}
1 & \Delta t \\
-\frac{k}{m}\Delta t & 1-\frac{c}{m}\Delta t
\end{bmatrix}}_{F_{1D}}\mathbf{x}_k +
\underbrace{\begin{bmatrix}
0 \\
\frac{\Delta t}{m}
\end{bmatrix}}_{G_{1D}}u_k,
\quad
y_k =
\underbrace{\begin{bmatrix}1 & 0\end{bmatrix}}_{H_{1D}}\mathbf{x}_k
$$

In [ ]:
# Part A: 1차원 Mass-Spring-Damper 가이드 예제

import numpy as np
import matplotlib.pyplot as plt

# ---- 시뮬레이션 시간 설정 ----
DT_1D = 0.01      # 샘플링 시간 [s]
T_END_1D = 10.0   # 종료 시간 [s]
t_1d = np.arange(0.0, T_END_1D + DT_1D, DT_1D)
N_1D = len(t_1d)  # 이산 스텝 수

# ---- MSD 시스템 물리 파라미터 ----
M = 1.0   # 질량 [kg]
C = 1.2   # 감쇠 계수 [N s/m]
K = 20.0  # 스프링 상수 [N/m]

# ---- 연속시간 모델: x_dot = A x + B u ----
A_1D = np.array([
    [XXXX, XXXX],
    [XXXX, XXXX]
])
B_1D = np.array([
    [XXXX],
    [XXXX]
])

# ---- 이산시간 모델: x[k+1] = F x[k] + G u[k] ----
F_1D = np.eye(2) + DT_1D * A_1D
G_1D = DT_1D * B_1D 
H_1D = np.array([[XXXX, XXXX]])  # 변위만 측정

print('F_1D =\n', F_1D)
print('G_1D =\n', G_1D)
print('H_1D =\n', H_1D)

# ---- 구간별 외력 입력 [N] ----
# +외력: 질량을 정방향으로 밀고, -외력: 반대 방향으로 당김
u_1d = np.zeros(N_1D)
u_1d[(t_1d >= 0.5) & (t_1d < 2.5)] = 5.0
u_1d[(t_1d >= 2.5) & (t_1d < 5.0)] = -3.0
u_1d[(t_1d >= 5.0)] = 0.0

# ---- 상태/출력 버퍼 할당 ----
# x_1d[0,:] = 변위, x_1d[1,:] = 속도
x_1d = np.zeros((2, N_1D))
y_1d = np.zeros(N_1D)

# 초기 상태: 변위=0 m, 속도=0 m/s
x_1d[:, 0] = [0.0, 0.0]
y_1d[0] = (H_1D @ x_1d[:, 0])[0]

# ---- 순방향 시뮬레이션 루프 ----
for k in range(1, N_1D):
    # 이전 스텝 입력 u[k-1]을 사용해 상태 전파
    x_1d[:, k] = F_1D @ x_1d[:, k - 1] + (G_1D[:, 0] * u_1d[k - 1])
    # 현재 상태로 측정 출력 계산
    y_1d[k] = (H_1D @ x_1d[:, k])[0]

print(f'final displacement = {y_1d[-1]:.3f} m')
print(f'final velocity     = {x_1d[1, -1]:.3f} m/s')

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(t_1d, u_1d, color='tab:orange')
axes[0].set_ylabel('u [N]')
axes[0].grid(True)

axes[1].plot(t_1d, x_1d[1, :], color='tab:blue')
axes[1].set_ylabel('v [m/s]')
axes[1].grid(True)

axes[2].plot(t_1d, y_1d, color='tab:green')
axes[2].set_ylabel('x [m]')
axes[2].set_xlabel('time [s]')
axes[2].grid(True)

fig.suptitle('Part A - 1D Mass-Spring-Damper (Guided)')
plt.tight_layout()
plt.show()

### Part A 확장 실습: 샘플링 시간 변화 실험

동일한 외력 입력에서 `\Delta t`를 변경하여 응답 차이를 비교한다.

- 실험 변수: `dt = 0.005, 0.01, 0.05`
- 비교 대상: 최종 변위, 최종 속도, 진동 파형
- 관찰 포인트: `dt` 증가 시 진동 해상도와 수치 안정성

In [ ]:
# Part A 확장: MSD 시스템 dt 스윕 실험

def run_msd_sim(dt, t_end=10.0, m=1.0, c=1.2, k=20.0):
    # 주어진 샘플링 시간에 대한 시간축 생성
    t = np.arange(0.0, t_end + dt, dt)
    n = len(t)

    # 연속시간 A, B를 거치지 않고 바로 이산 이행행렬 F, 입력행렬 G를 정의
    F = np.array([[XXXX, XXXX],
                  [XXXX, XXXX]])
    G = np.array([[XXXX],
                  [XXXX]])
    H = np.array([[XXXX, XXXX]])

    u = np.zeros(n)
    u[(t >= 0.5) & (t < 2.5)] = 5.0
    u[(t >= 2.5) & (t < 5.0)] = -3.0
    u[(t >= 5.0)] = 0.0

    x = np.zeros((2, n))
    y = np.zeros(n)
    x[:, 0] = [0.0, 0.0]
    y[0] = (H @ x[:, 0])[0]

    for idx in range(1, n):
        x[:, idx] = F @ x[:, idx - 1] + G[:, 0] * u[idx - 1]
        y[idx] = (H @ x[:, idx])[0]

    return t, u, x, y

DT_CANDIDATES = [0.005, 0.01, 0.05]
results_msd = {}

for dt in DT_CANDIDATES:
    # 동일한 시나리오를 dt만 바꿔 반복 실행
    t, u, x, y = run_msd_sim(dt)
    results_msd[dt] = (t, u, x, y)
    print(f'dt={dt:>5}: final displacement={y[-1]:8.4f} m, final velocity={x[1,-1]:8.4f} m/s')

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for dt in DT_CANDIDATES:
    t, u, x, y = results_msd[dt]
    axes[0].plot(t, x[1, :], label=f'dt={dt}')
    axes[1].plot(t, y, label=f'dt={dt}')

axes[0].set_ylabel('v [m/s]')
axes[0].grid(True)
axes[0].legend()
axes[0].set_title('Velocity comparison by dt (MSD)')

axes[1].set_ylabel('x [m]')
axes[1].set_xlabel('time [s]')
axes[1].grid(True)
axes[1].legend()
axes[1].set_title('Displacement comparison by dt (MSD)')

plt.tight_layout()
plt.show()

### Part A 확장 실습: 감쇠 조건별 응답 비교

질량 `m`과 스프링상수 `k`를 고정하고 감쇠계수 `c`를 변화시켜 응답 특성을 비교한다.

- Underdamped: `c < c_crit`
- Critically damped: `c = c_crit`
- Overdamped: `c > c_crit`

여기서 임계감쇠계수는 `c_crit = 2\sqrt{km}`이다.

In [ ]:
# Part A 확장: 감쇠 조건 비교

# 임계 감쇠를 기준으로 3가지 감쇠 구간 비교
m_cmp = 1.0
k_cmp = 20.0
c_crit = XXXX

cases = {
    'Underdamped': 0.4 * c_crit,
    'Critical': c_crit,
    'Overdamped': 2.0 * c_crit
}

# 과도응답 차이를 보기 위한 짧은 펄스 입력
u_cmp = np.zeros_like(t_1d)
u_cmp[(t_1d >= 0.5) & (t_1d < 1.0)] = 6.0

def run_msd_with_given_input(dt, t, u, m, c, k):
    A = np.array([[0.0, 1.0],
                  [-k / m, -c / m]])
    B = np.array([[0.0],
                  [1.0 / m]])
    F = np.eye(2) + dt * A
    G = dt * B

    x = np.zeros((2, len(t)))
    for idx in range(1, len(t)):
        x[:, idx] = F @ x[:, idx - 1] + G[:, 0] * u[idx - 1]
    return x

resp = {}
for name, c_val in cases.items():
    # 동일 입력 조건에서 감쇠계수만 바꿔 시뮬레이션
    x_case = run_msd_with_given_input(DT_1D, t_1d, u_cmp, m_cmp, c_val, k_cmp)
    resp[name] = x_case
    print(f'{name:12s}  c={c_val:7.3f}  final x={x_case[0,-1]:8.4f} m  final v={x_case[1,-1]:8.4f} m/s')

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for name, x_case in resp.items():
    axes[0].plot(t_1d, x_case[0, :], label=name)
    axes[1].plot(t_1d, x_case[1, :], label=name)

axes[0].set_ylabel('x [m]')
axes[0].set_title('Displacement response by damping condition')
axes[0].grid(True)
axes[0].legend()

axes[1].set_ylabel('v [m/s]')
axes[1].set_xlabel('time [s]')
axes[1].set_title('Velocity response by damping condition')
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

### Part A 분석: Damping 계수에 따른 Stability / Controllability / Observability

질량 `m`, 스프링상수 `k`, 샘플링 시간 `dt`를 고정하고 감쇠계수 `c`를 변화시키며 시스템 성질을 확인한다.

- Stability (asymptotic): 모든 고유값에 대해 `|λ| < 1`
- Controllability: 제어가능행렬 rank = 상태 차원
- Observability: 관측가능행렬 rank = 상태 차원

In [ ]:
# Part A 분석: 감쇠계수에 따른 시스템 성질

# -----
# 정의 설명
# - ctrb_mat(A, B): 주어진 시스템 행렬 A 및 입력 행렬 B에 대해, 연속/이산 시스템의 제어가능행렬(controllability matrix)를 계산합니다. 
#    이 행렬의 랭크가 상태공간 차원과 같으면 시스템은 완전히 제어 가능합니다.
# - obsv_mat(A, C): 관측가능행렬(observability matrix)을 계산합니다. C는 출력행렬입니다.
#    이 행렬의 랭크가 상태공간 차원과 같으면 시스템은 완전히 관측 가능합니다.
# - analyze_props_obsv(A, B, C): 고유값, 안정성, 제어 가능성, 관측 가능성을 한 번에 평가합니다.
# -----

def ctrb_mat(A, B):
    """
    제어가능행렬(controllability matrix) 계산 함수.
    Args:
        A: 상태행렬 (n x n)
        B: 입력행렬 (n x m)
    Returns:
        C: 제어가능행렬 (n x n*m)
    설명: [B, AB, A^2B, ..., A^{n-1}B]의 형태로 합친 행렬입니다.
    """
    n = A.shape[0]
    blocks = [B]
    Ak = np.eye(n)
    for _ in range(1, n):
        Ak = Ak @ A
        blocks.append(Ak @ B)
    return np.hstack(blocks)

def obsv_mat(A, C):
    """
    관측가능행렬(observability matrix) 계산 함수.
    Args:
        A: 상태행렬 (n x n)
        C: 출력행렬 (p x n)
    Returns:
        O: 관측가능행렬 (n*p x n)
    설명: [C; CA; CA^2; ...; CA^{n-1}] 형태로 쌓은 행렬입니다.
    """
    n = A.shape[0]
    blocks = [C]
    Ak = np.eye(n)
    for _ in range(1, n):
        Ak = Ak @ A
        blocks.append(C @ Ak)
    return np.vstack(blocks)

def analyze_props_obsv(A, B, C, tol=1e-9):
    """
    시스템의 주요 성질 분석 함수. (관측가능성 포함)
    Args:
        A: 상태행렬 (n x n)
        B: 입력행렬 (n x m)
        C: 출력행렬 (p x n)
        tol: 허용 오차
    Returns:
        eigvals: 고유값 벡터
        stable: (bool) 모든 고유값이 1보다 작으면 True (안정)
        controllable: (bool) 제어가능성 (ctrb 행렬 랭크로 판단)
        observable: (bool) 관측가능성 (obsv 행렬 랭크로 판단)
    """
    eigvals = np.linalg.eigvals(A)
    stable = np.all(np.abs(eigvals) < 1.0 - tol)
    rank_ctrb = np.linalg.matrix_rank(ctrb_mat(A, B), tol)
    controllable = (rank_ctrb == A.shape[0])
    rank_obsv = np.linalg.matrix_rank(obsv_mat(A, C), tol)
    observable = (rank_obsv == A.shape[0])
    return eigvals, stable, controllable, observable

# 관측행렬 C 정의: 위치만 측정
C_STD = np.array([1.0, 0.0])

c_values = np.linspace(0.0, 25.0, 11)
max_abs_eig = []

print('c value sweep (MSD):')
print(' c      max|lambda|   stable   controllable   observable')
print('-' * 60)
for c_val in c_values:
    # 감쇠계수 c마다 (F, G) 행렬 재구성
    # A_tmp = np.array([[0.0, 1.0],
    #                   [-K / M, -c_val / M]])
    # B_tmp = np.array([[0.0],
    #                   [1.0 / M]])
    # F_tmp = np.eye(2) + DT_1D * A_tmp
    # G_tmp = DT_1D * B_tmp
    F_tmp = np.array([[           0.0,              DT_1D], 
                      [-(K * DT_1D)/M, -(c_val * DT_1D)/M]])
    G_tmp = np.array([[0.0],
                      [DT_1D / M]])

    # 3가지 성질(안정성/제어가능성/관측가능성) 평가
    eigvals, stable, controllable, observable = analyze_props_obsv(F_tmp, G_tmp, C_STD)
    max_eig = np.max(np.abs(eigvals))
    max_abs_eig.append(max_eig)

    print(f'{c_val:5.2f}   {max_eig:10.6f}   {str(stable):>6s}   {str(controllable):>12s}   {str(observable):>12s}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(c_values, max_abs_eig, marker='o')
ax.axhline(1.0, color='r', linestyle='--', label='|lambda|=1 boundary')
ax.set_xlabel('damping coefficient c [N s/m]')
ax.set_ylabel('max |lambda(F)|')
ax.set_title('MSD stability trend with damping coefficient')
ax.grid(True)
ax.legend()
plt.tight_layout()
plt.show()

## Part B. 스스로 해보기: 2차원 CA 모델


### 문제 설명

본 파트는 **자기 주도 실습 구간**이다.

시뮬레이션 대상은 2차원 공간(X, Y)에서 이동하는 차량의 동적 상태 벡터(위치, 속도)이며, 원점에 위치한 거리 센서(range sensor)가 차량의 위치를 측정한다.

**학습 과제**
- 상태 벡터 및 입력 벡터의 차원을 점검한다.
- `F`, `G`, `H` 행렬의 각 원소가 반영하는 물리량을 설명한다.
- Ground Truth와 시뮬레이션 출력을 비교한다.


<img src="../resources/ch10/range_sensor.png" width="50%">

### 운동 모델 - 등가속도 운동 (CA)

가속도 입력을 갖는 이산 차량 운동 모델은 다음과 같이 표현됩니다:


등가속도(2차원) 운동 모델의 상태 공간 표현:
$$
\begin{align*}
\begin{bmatrix}
x_{t+1} \\
y_{t+1} \\
v_{x,{t+1}} \\
v_{y,{t+1}}
\end{bmatrix}
=
\begin{bmatrix}
x_t + v_{x,t}\,dT + \frac{1}{2} a_{x,t}\,dT^2 \\
y_t + v_{y,t}\,dT + \frac{1}{2} a_{y,t}\,dT^2 \\
v_{x,t} + a_{x,t}\,dT \\
v_{y,t} + a_{y,t}\,dT
\end{bmatrix}
\end{align*}
$$

여기서 $dT$는 샘플링 시간 간격입니다.

- x_t, y_t : t 시점에서 차량의 X, Y **위치**
- v_x_t, v_y_t : t 시점에서 차량의 X, Y **속도**
- a_x_t, a_y_t : t 시점에서 차량의 X, Y **가속도 입력**
- dT: **샘플링 시간**


### 상태 공간 방정식 (State Space Equation)


#### 시스템 시간 전파 (System time propagation)


시스템의 상태 전파 식:
$$
\begin{align*}
\begin{bmatrix}
x_{t+1} \\
y_{t+1} \\
v_{x,{t+1}} \\
v_{y,{t+1}}
\end{bmatrix}
=
\underbrace{
\begin{bmatrix}
1 & 0 & dT & 0 \\
0 & 1 & 0 & dT \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
}_{F}
\begin{bmatrix}
x_t \\
y_t \\
v_{x,t} \\
v_{y,t}
\end{bmatrix}
+
\underbrace{
\begin{bmatrix}
\frac{1}{2}dT^2 & 0 \\
0 & \frac{1}{2}dT^2 \\
dT & 0 \\
0 & dT
\end{bmatrix}
}_{G}
\begin{bmatrix}
a_{x,t} \\
a_{y,t}
\end{bmatrix}
\end{align*}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── 시뮬레이션 파라미터 설정 ──
delta_t   = 0.1   # 샘플링 시간 [sec]
start_t   = 0.0   # 시작 시간 [sec]
end_t     = 30.0  # 종료 시간 [sec]

time                = np.arange(start_t, end_t + delta_t, delta_t)
total_process_steps = len(time)  # 301 스텝

print(f'샘플링 시간         : {delta_t} s')
print(f'총 시뮬레이션 스텝  : {total_process_steps}')
print(f'시간 범위           : {time[0]} ~ {time[-1]} s')

# ────────────────────────────────────────────
# [실습] 아래 행렬을 채우세요
# ────────────────────────────────────────────

# 시스템 행렬 F (4x4): 이전 상태 -> 현재 상태 전파
F = np.array(XXXXMatrixXXXX)

# 제어 행렬 G (4x2): 가속도 입력 -> 상태 변화량
G = np.array(XXXXMatrixXXXX)

print('\nF =\n', F)
print('\nG =\n', G)


#### 출력 방정식 (Output)


$$
\mathbf{y}_k =
\underbrace{
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0
\end{bmatrix}
}_{H}
\mathbf{x}_k
$$

In [ ]:
# ────────────────────────────────────────────
# [실습] 측정 행렬 H를 채우세요
# ────────────────────────────────────────────

# 측정 행렬 H (2x4): 상태 벡터에서 위치(x, y)만 추출
H = np.array(XXXXMatrixXXXX)

print('H =\n', H)


## 시뮬레이션 설정


### 초기 조건

- Δt: 0.1초
- 시뮬레이션 시간: 0 ~ 30초


In [ ]:
# ── 상태 배열 초기화 ──
# acceleration : shape (2, N)  [0번 행: x방향,  1번 행: y방향]  단위: m/s²
# velocity     : shape (2, N)  [0번 행: x방향,  1번 행: y방향]  단위: m/s
# position     : shape (2, N)  [0번 행: x방향,  1번 행: y방향]  단위: m

acceleration = np.zeros((2, total_process_steps))
velocity     = np.zeros((2, total_process_steps))
position     = np.zeros((2, total_process_steps))

print('배열 초기화 완료')
print(f'  acceleration : {acceleration.shape}')
print(f'  velocity     : {velocity.shape}')
print(f'  position     : {position.shape}')


### 시뮬레이션 - CA 선형 모델 (Ground Truth 데이터 생성)


In [ ]:
# ── Ground Truth 데이터 생성 (가속도 / 속도 / 위치) ──
#
# 가속도 프로파일:
#   0  <= t < 10 : ax = ay = 2  m/s²  (가속 구간)
#   10 <= t < 20 : ax = ay = 4  m/s²  (고가속 구간)
#   20 <= t <= 30: ax = ay = 2  m/s²  (감속 구간)

velocity[:, 0] = [0.0, 0.0]  # 초기 속도: vx=0, vy=0 [m/s]
position[:, 0] = [1.0, 1.0]  # 초기 위치: x=1,  y=1  [m]

for idx in range(total_process_steps):
    t = time[idx]
    if t < 10:
        acceleration[:, idx] = [2, 2]
    elif t < 20:
        acceleration[:, idx] = [4, 4]
    else:
        acceleration[:, idx] = [2, 2]

    if idx > 0:
        velocity[:, idx] = velocity[:, idx-1] + acceleration[:, idx-1] * delta_t
        position[:, idx] = (position[:, idx-1]
                            + velocity[:, idx-1] * delta_t
                            + 0.5 * acceleration[:, idx-1] * delta_t**2)

print('Ground Truth 데이터 생성 완료')
print(f'최종 위치 : x = {position[0,-1]:.2f} m,  y = {position[1,-1]:.2f} m')
print(f'최종 속도 : vx = {velocity[0,-1]:.2f} m/s, vy = {velocity[1,-1]:.2f} m/s')


## Ground Truth 상태 플롯


**실제 가속도 (true acceleration)**


<img src="../resources/ch10/true_acceleration.png" width="50%">

**실제 속도 (true velocity)**


<img src="../resources/ch10/true_velocity.png" width="50%">

**실제 위치 (true position)**


<img src="../resources/ch10/true_position.png" width="50%">

**이동 궤적 (trajectory)**


<img src="../resources/ch10/trajectory.png" width="50%">

In [ ]:
# ── Figure 1: 실제 가속도 ──
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
fig.suptitle('Constant Acceleration Model - 가속도 (acceleration)')
axes[0].plot(time, acceleration[0, :], 'b', label='a_x')
axes[0].set_xlabel('시간 [s]'); axes[0].set_ylabel('가속도 [m/s²]')
axes[0].set_ylim([0, 6]); axes[0].legend(); axes[0].grid(True)
axes[1].plot(time, acceleration[1, :], 'b', label='a_y')
axes[1].set_xlabel('시간 [s]'); axes[1].set_ylabel('가속도 [m/s²]')
axes[1].set_ylim([0, 6]); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

# ── Figure 2: 실제 속도 ──
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
fig.suptitle('Constant Acceleration Model - 속도 (velocity)')
axes[0].plot(time, velocity[0, :], 'b', label='v_x')
axes[0].set_xlabel('시간 [s]'); axes[0].set_ylabel('속도 [m/s]')
axes[0].legend(); axes[0].grid(True)
axes[1].plot(time, velocity[1, :], 'b', label='v_y')
axes[1].set_xlabel('시간 [s]'); axes[1].set_ylabel('속도 [m/s]')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

# ── Figure 3: 실제 위치 ──
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
fig.suptitle('Constant Acceleration Model - 위치 (position)')
axes[0].plot(time, position[0, :], 'b', label='x')
axes[0].set_xlabel('시간 [s]'); axes[0].set_ylabel('x [m]')
axes[0].legend(); axes[0].grid(True)
axes[1].plot(time, position[1, :], 'b', label='y')
axes[1].set_xlabel('시간 [s]'); axes[1].set_ylabel('y [m]')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

# ── Figure 4: 이동 궤적 ──
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(position[0, :], position[1, :], 'b', label='궤적')
ax.scatter(position[0, 0],  position[1, 0],  c='g', s=80, zorder=5, label='시작')
ax.scatter(position[0, -1], position[1, -1], c='r', s=80, zorder=5, label='끝')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.set_title('이동 궤적 (Trajectory)'); ax.legend(); ax.grid(True)
ax.set_aspect('equal'); plt.tight_layout(); plt.show()


## 0 ~ 30초 동안 상태 및 측정값 시뮬레이션


### 초기 조건

- Δt: 0.1초
- 초기 속도: vx = 0.0 m/s, vy = 0.0 m/s
- 초기 위치: x = 1 m, y = 1 m
- 시뮬레이션 시간: 0 ~ 30초


In [ ]:
STATE_ORDER  = 4  # 상태 벡터 차원 [위치 x, 위치 y, 속도 x, 속도 y]
OUTPUT_ORDER = 2  # 출력 벡터 차원 [위치 x, 위치 y]

state  = np.zeros((STATE_ORDER,  total_process_steps))
output = np.zeros((OUTPUT_ORDER, total_process_steps))

isFirstStep = True

for idx in range(total_process_steps):
    if isFirstStep:
        # ────────────────────────────────────────────
        # [실습] 초기 상태를 채우세요
        # ────────────────────────────────────────────
        state[0, idx] = XXXX   # 초기 위치 x [m]
        state[1, idx] = XXXX   # 초기 위치 y [m]
        state[2, idx] = XXXX   # 초기 속도 x [m/s]
        state[3, idx] = XXXX   # 초기 속도 y [m/s]
        isFirstStep = False
    else:
        # ────────────────────────────────────────────
        # [실습] F, G 행렬과 입력을 이용해 상태를 갱신하세요
        # state[:, idx] = ?
        # ────────────────────────────────────────────
        state[:, idx] = XXXXXXXXXX

    # ────────────────────────────────────────────
    # [실습] H 행렬을 이용해 출력(측정값)을 계산하세요
    # output[:, idx] = ?
    # ────────────────────────────────────────────
    output[:, idx] = XXXXXXXXX

print('시뮬레이션 완료')
print(f'최종 상태  : {state[:, -1]}')
print(f'최종 출력  : {output[:, -1]}')


## 결과 시각화


## 오차 분석 (GT vs Simulation)

다음 지표를 계산하여 모델 구현 결과를 정량적으로 확인한다.

- x축 위치 RMSE
- y축 위치 RMSE
- 최대 절대 오차 (x, y)

In [ ]:
# Quantitative error metrics

error_x = output[0, :] - position[0, :]
error_y = output[1, :] - position[1, :]

rmse_x = np.sqrt(np.mean(error_x**2))
rmse_y = np.sqrt(np.mean(error_y**2))
max_abs_x = np.max(np.abs(error_x))
max_abs_y = np.max(np.abs(error_y))

print('Error metrics (GT vs Simulation)')
print(f'RMSE x      : {rmse_x:.6f} m')
print(f'RMSE y      : {rmse_y:.6f} m')
print(f'Max |error| x: {max_abs_x:.6f} m')
print(f'Max |error| y: {max_abs_y:.6f} m')

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(time, error_x, label='error_x', color='tab:red')
axes[0].set_ylabel('error x [m]')
axes[0].grid(True)
axes[0].legend()

axes[1].plot(time, error_y, label='error_y', color='tab:purple')
axes[1].set_ylabel('error y [m]')
axes[1].set_xlabel('time [s]')
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 1: 시뮬레이션된 위치 ──
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
fig.suptitle('Constant Acceleration Model - 추정 위치 (Estimate Position)')
axes[0].plot(time, output[0, :], 'g', label='x')
axes[0].set_xlabel('시간 [s]'); axes[0].set_ylabel('x [m]')
axes[0].legend(); axes[0].grid(True)
axes[1].plot(time, output[1, :], 'g', label='y')
axes[1].set_xlabel('시간 [s]'); axes[1].set_ylabel('y [m]')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

# ── Figure 2: Ground Truth vs 시뮬레이션 비교 ──
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
fig.suptitle('실제 위치 vs 추정 위치 비교 (GT vs Estimate)')
axes[0].plot(time, position[0, :], 'b', label='GT')
axes[0].plot(time, output[0, :],   'g', label='Estimate')
axes[0].set_xlabel('시간 [s]'); axes[0].set_ylabel('x [m]')
axes[0].legend(); axes[0].grid(True)
axes[1].plot(time, position[1, :], 'b', label='GT')
axes[1].plot(time, output[1, :],   'g', label='Estimate')
axes[1].set_xlabel('시간 [s]'); axes[1].set_ylabel('y [m]')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()


## Part B 분석: Stability, Controllability, Observability

이 절에서는 Part B의 2D CA 모델
\(\mathbf{x}_{k+1}=F\mathbf{x}_k+G\mathbf{u}_k\)
에 대해 다음 항목을 점검한다.

- Stability (asymptotic): 모든 고유값에 대해 `|λ| < 1`
- Controllability: 제어가능행렬 rank = 상태 차원
- Observability: 관측가능행렬 rank = 상태 차원

In [ ]:
# Part B property analysis (2D CA model)

def controllability_matrix(A, B):
    n = A.shape[0]
    blocks = [B]
    Ak = np.eye(n)
    for _ in range(1, n):
        Ak = Ak @ A
        blocks.append(Ak @ B)
    return np.hstack(blocks)

def observability_matrix(A, C):
    n = A.shape[0]
    blocks = [C]
    Ak = np.eye(n)
    for _ in range(1, n):
        Ak = Ak @ A
        blocks.append(C @ Ak)
    return np.vstack(blocks)

eigvals_ca = np.linalg.eigvals(F)
stable_ca = np.all(np.abs(eigvals_ca) < 1.0 - 1e-9)
ctrb_ca = controllability_matrix(F, G)
rank_ctrb_ca = np.linalg.matrix_rank(ctrb_ca, 1e-9)
controllable_ca = (rank_ctrb_ca == F.shape[0])

obsv_ca = observability_matrix(F, H)
rank_obsv_ca = np.linalg.matrix_rank(obsv_ca, 1e-9)
observable_ca = (rank_obsv_ca == F.shape[0])

print('===== 2D Constant Acceleration =====')
print('eigenvalues(F):', np.round(eigvals_ca, 6))
print(f'Stability (|lambda|<1): {stable_ca}')
print(f'Controllability rank: {rank_ctrb_ca}/{F.shape[0]} -> {controllable_ca}')
print(f'Observability rank: {rank_obsv_ca}/{F.shape[0]} -> {observable_ca}')